# Getting Started with Foundry Agent Service

By the end of this notebook, you'll have built a **Foundry Expert** — an AI agent that can answer questions from Microsoft Learn documentation, review SDK source code on GitHub, search the web for the latest announcements, execute Python code, and recommend models from your Foundry project. Then you'll publish it to Microsoft 365 Copilot and Teams.

## What you'll build

We'll start simple — a name, instructions, and a model — then layer on capabilities one at a time:

1. **Create** your agent with a name, instructions, and a model
2. **Give it knowledge** — connect it to Microsoft Learn documentation via MCP
3. **Give it eyes** — add web search for real-time information
4. **Give it hands** — add Code Interpreter so it can write and run Python
5. **Give it expertise** — connect it to the Azure SDK on GitHub via MCP
6. **Give it judgment** — connect it to your Foundry project via the Foundry MCP server
7. **Combine everything** — build a multi-tool agent and stream the response
8. **Publish** — deploy your agent to Microsoft 365 Copilot and Teams

## Prerequisites

- An Azure subscription ([create one free](https://azure.microsoft.com/pricing/purchase-options/azure-account))
- A [Microsoft Foundry project](https://ai.azure.com) with a deployed model (e.g., `gpt-4.1`, `gpt-5.2`)
- Python 3.10+
- Azure CLI installed and signed in (`az login`)

---

## 1. What is Foundry Agent Service?

Foundry Agent Service is the managed runtime at the heart of Microsoft Foundry. Let's understand what it gives you before we start building.

Here's what it handles for you:

| Capability | What it does |
|---|---|
| **Conversation management** | Maintains multi-turn state across sessions |
| **Tool orchestration** | Routes to Code Interpreter, File Search, Bing, custom functions, MCP servers, and more |
| **Content safety** | Enforces responsible AI guardrails by default |
| **Identity & RBAC** | Integrates with Microsoft Entra ID for secure, least-privilege access |
| **Observability** | Built-in tracing with Application Insights and OpenTelemetry |
| **Scale** | Auto-scales from prototype to production without infrastructure changes |

### What makes up an agent?

Think of an agent as having six building blocks:

- **Name** — what it is (its identity)
- **Instructions** — how it should behave (goals, tone, constraints)
- **Model** — the reasoning engine it thinks with. Choosing a model is like choosing which expert sits in the driver's seat — a faster, cheaper model for simple tasks, a more capable one for complex reasoning.
- **Knowledge** — what it knows (file search, grounding data)
- **Memory** — what it remembers (conversations, state across turns)
- **Tools** — what it can *do* (run code, call APIs, search the web)

In this notebook, we'll start simple — just a name, instructions, and a model — then layer on the rest as we go.

### Architecture at a glance

```
┌─────────────────────────────────────────────────────┐
│                  Your Application                    │
│   AIProjectClient  →  OpenAI Client (Responses API) │
└──────────────┬──────────────────────┬───────────────┘
               │                      │
     ┌─────────▼─────────┐  ┌────────▼────────┐
     │  Agent Definitions │  │  Conversations   │
     │  (PromptAgent /    │  │  (Multi-turn     │
     │   WorkflowAgent)   │  │   state mgmt)    │
     └─────────┬─────────┘  └────────┬────────┘
               │                      │
     ┌─────────▼──────────────────────▼───────┐
     │        Foundry Agent Service Runtime    │
     │  ┌──────────┐ ┌────────┐ ┌───────────┐ │
     │  │Code Interp│ │File    │ │Bing/Web   │ │
     │  │          │ │Search  │ │Search     │ │
     │  └──────────┘ └────────┘ └───────────┘ │
     │  ┌──────────┐ ┌────────┐ ┌───────────┐ │
     │  │Functions │ │MCP     │ │SharePoint │ │
     │  │          │ │Servers │ │           │ │
     │  └──────────┘ └────────┘ └───────────┘ │
     └────────────────────────────────────────┘
```

---

## 2. Set Up Your Environment

Before we write any code, let's make sure you have access.

### Required RBAC permissions

| Action | Required Role |
|---|---|
| Create a Foundry account and project | **Azure AI Account Owner** |
| Create and edit agents (SDK / Playground) | **Azure AI User** |
| Deploy models | **Azure AI Project Manager** |
| Assign RBAC for Standard setup resources | **Role Based Access Control Administrator** |

> **Tip:** If you're just getting started, ask your admin for **Azure AI User** on your project — that's all you need for everything in this notebook.

### Install the SDK

The Foundry SDK v2 gives you two clients that work together:
- `AIProjectClient` — manages agents, connections, and project resources
- `OpenAI client` (from project) — sends messages and handles conversations

In [ ]:
%pip install azure-ai-projects --pre azure-identity python-dotenv openai --quiet

### Configure environment variables

Create a `.env` file in your project root with the following values:

```env
FOUNDRY_PROJECT_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/api/projects/<your-project>
FOUNDRY_MODEL_DEPLOYMENT_NAME=gpt-5.2
FOUNDRY_GITHUB_CONNECTION=github-mcp
FOUNDRY_CODE_INTERPRETER_URL=https://<your-container-app>.azurecontainerapps.io
FOUNDRY_CODE_INTERPRETER_CONNECTION=<your-mcp-connection-id>
```

You can find your **Project Endpoint** in the [Foundry portal](https://ai.azure.com) under your project's overview page.

| Variable | Where to find it |
|---|---|
| `FOUNDRY_PROJECT_ENDPOINT` | Foundry portal → Project → Overview |
| `FOUNDRY_MODEL_DEPLOYMENT_NAME` | Your deployed model name (e.g., `gpt-5.2`, `gpt-4.1`) |
| `FOUNDRY_GITHUB_CONNECTION` | Foundry portal → Connections → Custom Keys (see Section 8) |
| `FOUNDRY_CODE_INTERPRETER_URL` | Azure portal → Container App → Application URL (see Section 7) |
| `FOUNDRY_CODE_INTERPRETER_CONNECTION` | Foundry portal → Connections → your MCP connection ID |

> **Note:** The custom code interpreter and GitHub MCP connections require one-time setup. We'll walk you through each when we get there.

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()

# Initialize the project client
project_client = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)

# Get the OpenAI client from the project
openai_client = project_client.get_openai_client()

MODEL = os.environ.get("FOUNDRY_MODEL_DEPLOYMENT_NAME", "gpt-5.2")
print(f"✅ Connected to project: {os.environ['FOUNDRY_PROJECT_ENDPOINT']}")
print(f"✅ Using model: {MODEL}")

**Expected output:**
```
✅ Connected to project: https://<your-resource>.services.ai.azure.com/api/projects/<your-project>
✅ Using model: gpt-5.2
```

---

## 3. Create Your First Agent

Let's create your first agent. You give it three things: a **name** (what it is), **instructions** (how it should behave), and a **model** (the reasoning engine it thinks with). That's it — three lines and you have an agent.

We're calling ours `foundry-expert` — an agent that knows Microsoft Foundry and helps developers build with it.

> **About memory:** Agents can also have *memory* — the ability to remember context across conversations. We won't build that into this tutorial, but as your agent matures, the Conversations API lets you add persistent, multi-turn memory. Think of it as a capability you unlock later.

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

# Create the Foundry Expert agent
agent = project_client.agents.create_version(
    agent_name="foundry-expert",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a Foundry Expert — a knowledgeable assistant for "
            "Microsoft Foundry (Azure AI Foundry). You help developers "
            "understand the platform, write code using the azure-ai-projects "
            "SDK, find the latest announcements, and recommend the right "
            "tools and models for their workloads. "
            "Be concise, accurate, and friendly."
        ),
    ),
)

print(f"✅ Agent created")
print(f"   Name:    {agent.name}")
print(f"   Version: {agent.version}")
print(f"   ID:      {agent.id}")

**Expected output:**
```
✅ Agent created
   Name:    foundry-expert
   Version: 1
   ID:      asst_abc123xyz
```

---

## 4. Chat with Your Agent

Now let's talk to it. You use the OpenAI-compatible Responses API — same patterns you might already know, but running against your Foundry project.

In [ ]:
# Ask a Foundry-specific question
response = openai_client.responses.create(
    extra_body={"agent": {"name": "foundry-expert", "type": "agent_reference"}},
    input="What's the difference between a PromptAgentDefinition and a WorkflowAgentDefinition?",
)

print(response.output_text)

The agent answered — but it's working from its training data. It might be slightly outdated or miss nuances from the latest documentation. Let's fix that by giving it **knowledge** — real-time access to Microsoft Learn docs.

---

## 5. Give It Knowledge — Microsoft Learn MCP

The [Model Context Protocol (MCP)](https://learn.microsoft.com/azure/ai-foundry/agents/how-to/tools/model-context-protocol) lets you connect your agent to external tool servers. The [Microsoft Learn MCP server](https://learn.microsoft.com/training/support/mcp) gives your agent direct access to official Microsoft documentation — always current, always authoritative.

In [ ]:
from azure.ai.projects.models import MCPTool

# Connect to the official Microsoft Learn MCP server
docs_tool = MCPTool(
    server_label="microsoft-learn",
    server_url="https://learn.microsoft.com/api/mcp",
    require_approval="never",
)

# Upgrade the agent with documentation access
docs_agent = project_client.agents.create_version(
    agent_name="foundry-expert-docs",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a Foundry Expert with access to Microsoft Learn documentation. "
            "When answering questions about Microsoft Foundry, Azure AI, or the "
            "azure-ai-projects SDK, search the documentation for accurate, up-to-date "
            "information. Always cite your sources."
        ),
        tools=[docs_tool],
    ),
)
print(f"✅ Agent created with Microsoft Learn access (version: {docs_agent.version})")

In [ ]:
# Ask something that requires current documentation
response = openai_client.responses.create(
    extra_body={"agent": {"name": "foundry-expert-docs", "type": "agent_reference"}},
    input="What RBAC roles are needed to publish an agent to Microsoft Teams?",
)

print(response.output_text)

The agent searched Microsoft Learn and returned an answer grounded in official documentation — not guesswork. But what about information that isn't in the docs yet, like recent announcements or blog posts?

---

## 6. Give It Eyes — Web Search

The `WebSearchPreviewTool` gives your agent access to the live web — no Bing connection or additional setup required.

In [ ]:
from azure.ai.projects.models import WebSearchPreviewTool

# Add web search capability
web_tool = WebSearchPreviewTool()

web_agent = project_client.agents.create_version(
    agent_name="foundry-expert-web",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a Foundry Expert with access to web search. "
            "Use web search to find the latest announcements, blog posts, "
            "and event coverage about Microsoft Foundry and Azure AI. "
            "Always cite your sources with URLs."
        ),
        tools=[web_tool],
    ),
)
print(f"✅ Agent created with web search (version: {web_agent.version})")

In [ ]:
# Ask about something recent that docs might not cover yet
response = openai_client.responses.create(
    extra_body={"agent": {"name": "foundry-expert-web", "type": "agent_reference"}},
    input="What was announced about Foundry Agent Service at Microsoft Build 2026?",
)

print(response.output_text)

Now your agent can find information from both documentation and the live web. But what if you need it to actually *do* something — run code, analyze data, test an idea?

---

## 7. Give It Hands — Custom Code Interpreter

The built-in Code Interpreter is convenient, but it ships with a fixed set of Python packages. If your agent needs to run code with specific dependencies — like `azure-ai-projects` itself, `pandas`, or anything your team uses — you need a **custom code interpreter**.

A [custom code interpreter](https://learn.microsoft.com/azure/ai-foundry/agents/how-to/tools/custom-code-interpreter) is an MCP server backed by Azure Container Apps. You control the Python packages, the compute resources, and the runtime environment. The agent sends code to it just like the built-in version, but now it executes in *your* container with *your* dependencies.

### One-time setup

If you haven't already provisioned the custom code interpreter infrastructure:

1. Clone the [foundry-samples](https://github.com/azure-ai-foundry/foundry-samples) repo
2. Navigate to `samples/python/hosted-agents/code-interpreter-custom`
3. Deploy with `az deployment group create --template-file ./infra.bicep`
4. Copy the MCP server URL and connection ID into your `.env` file

See the [full setup guide](https://learn.microsoft.com/azure/ai-foundry/agents/how-to/tools/custom-code-interpreter) for details.

In [ ]:
import os

# Custom code interpreter — your own Python runtime via MCP
code_tool = MCPTool(
    server_label="custom-code-interpreter",
    server_url=os.environ["FOUNDRY_CODE_INTERPRETER_URL"],
    project_connection_id=os.environ.get("FOUNDRY_CODE_INTERPRETER_CONNECTION"),
)

code_agent = project_client.agents.create_version(
    agent_name="foundry-expert-code",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a Foundry Expert that can write and execute Python code. "
            "Your runtime has azure-ai-projects, pandas, matplotlib, and other "
            "common data science packages pre-installed. When asked to demonstrate "
            "SDK usage, write working code and run it. Show the output."
        ),
        tools=[code_tool],
    ),
)
print(f"✅ Agent created with custom Code Interpreter (version: {code_agent.version})")

In [ ]:
# Ask the agent to write and execute real SDK code
response = openai_client.responses.create(
    extra_body={"agent": {"name": "foundry-expert-code", "type": "agent_reference"}},
    input=(
        "Write a Python script using azure-ai-projects that lists all model "
        "deployments in my project, showing the model name, publisher, and SKU "
        "for each. Run it and show the output."
    ),
)

print(response.output_text)

The agent wrote Python code using the `azure-ai-projects` SDK, executed it in your custom container — with all the right packages pre-installed — and returned real results from your project. That's the power of a custom code interpreter: your agent runs code in *your* environment, not a generic sandbox.

---

## 8. Give It Expertise — GitHub MCP

What if your agent could read the actual SDK source code? The [GitHub MCP server](https://github.com/github/github-mcp-server) gives your agent read access to repositories, issues, and pull requests.

> **Note:** The GitHub MCP server requires authentication via a project connection. In the Foundry portal, create a **Custom Keys** connection with key `Authorization` and value `Bearer <your-github-pat>`. See [MCP authentication](https://learn.microsoft.com/azure/ai-foundry/agents/how-to/mcp-authentication) for details.

In [ ]:
import os

# Connect to the official GitHub MCP server
# Requires a project connection with GitHub PAT authentication
# Set up in Foundry portal: Connections > Custom Keys > key "Authorization", value "Bearer <PAT>"

github_connection = project_client.connections.get(
    os.environ["FOUNDRY_GITHUB_CONNECTION"]
)

github_tool = MCPTool(
    server_label="github",
    server_url="https://api.githubcopilot.com/mcp/",
    require_approval="never",
    project_connection_id=github_connection.id,
)

github_agent = project_client.agents.create_version(
    agent_name="foundry-expert-github",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a Foundry Expert with access to GitHub repositories. "
            "You can read source code, issues, and pull requests from the "
            "Azure SDK for Python. Use this to answer questions about SDK "
            "implementation details and recent changes."
        ),
        tools=[github_tool],
    ),
)
print(f"✅ Agent created with GitHub access (version: {github_agent.version})")

In [ ]:
# Ask the agent to look at real SDK source code
response = openai_client.responses.create(
    extra_body={"agent": {"name": "foundry-expert-github", "type": "agent_reference"}},
    input=(
        "Look at the azure-ai-projects SDK in the Azure/azure-sdk-for-python repo. "
        "What tools are available for agents? Show me the class names and a brief "
        "description of each."
    ),
)

print(response.output_text)

Your agent just read the actual SDK source code on GitHub and summarized what it found. That's like having a code reviewer on demand.

---

## 9. Give It Judgment — Foundry MCP Server

Here's where it gets really interesting. The [Foundry MCP server](https://learn.microsoft.com/azure/ai-foundry/mcp/get-started) connects your agent to *your actual Foundry project*. It can list deployments, query agents, and interact with project resources — giving your agent the ability to make informed recommendations based on what's really available.

We set `require_approval="always"` here because the agent is accessing real project resources. You'll see the approval flow in action.

In [ ]:
import os

# Connect to the Foundry MCP server — your project's own tool endpoint
foundry_tool = MCPTool(
    server_label="foundry-project",
    server_url=os.environ["FOUNDRY_PROJECT_ENDPOINT"] + "/mcp_tools?api-version=2025-05-15-preview",
    require_approval="always",  # Require approval for project resource access
)

foundry_agent = project_client.agents.create_version(
    agent_name="foundry-expert-project",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a Foundry Expert with direct access to the user's Foundry project. "
            "You can list model deployments, check capabilities, and make recommendations "
            "based on what's actually available. When recommending models, consider the "
            "user's requirements for cost, speed, and capability."
        ),
        tools=[foundry_tool],
    ),
)
print(f"✅ Agent created with Foundry project access (version: {foundry_agent.version})")

In [ ]:
from openai.types.responses.response_input_param import McpApprovalResponse

# Ask the agent to recommend a model based on real deployments
response = openai_client.responses.create(
    extra_body={"agent": {"name": "foundry-expert-project", "type": "agent_reference"}},
    input=(
        "Look at the model deployments available in my project. "
        "Which model would you recommend for a cost-sensitive customer "
        "support chatbot, and why?"
    ),
)

# Handle MCP approval requests
input_list = []
for item in response.output:
    if item.type == "mcp_approval_request":
        print(f"🔐 Approving tool call: {item.name} on {item.server_label}")
        input_list.append(
            McpApprovalResponse(
                type="mcp_approval_response",
                approve=True,
                approval_request_id=item.id,
            )
        )

# If there were approval requests, send them and get the final response
if input_list:
    response = openai_client.responses.create(
        input=input_list,
        previous_response_id=response.id,
        extra_body={"agent": {"name": "foundry-expert-project", "type": "agent_reference"}},
    )

print(response.output_text)

The agent looked at your *actual* model deployments and made a recommendation based on real data — not hypothetical comparisons. Notice the approval flow: because we set `require_approval="always"`, you get to review every tool call before it executes. This is important when your agent accesses real project resources.

---

## 10. Put It All Together

Here's where it gets powerful. Let's combine all five tools into a single agent and let the model's reasoning decide which tool to use for each question. Ask about docs? Microsoft Learn. Ask about latest news? Web search. Ask about code? GitHub. Need a calculation? Code Interpreter. Need project-specific info? Foundry MCP.

In [ ]:
# Build the ultimate Foundry Expert with ALL tools
expert = project_client.agents.create_version(
    agent_name="foundry-expert-full",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are the Foundry Expert — a comprehensive AI assistant for "
            "Microsoft Foundry developers. You have access to:\n"
            "- Microsoft Learn documentation (search official docs)\n"
            "- Web search (find latest announcements and blog posts)\n"
            "- Custom Code Interpreter (write and run Python with azure-ai-projects, pandas, etc.)\n"
            "- GitHub (read Azure SDK source code and issues)\n"
            "- Foundry Project (list deployments, recommend models)\n\n"
            "Choose the right tool for each question. Be thorough, cite "
            "sources, and provide actionable guidance."
        ),
        tools=[docs_tool, web_tool, code_tool, github_tool, foundry_tool],
    ),
)

print(f"✅ Foundry Expert created with {len(expert.definition.tools)} tools")
print(f"   Name:    {expert.name}")
print(f"   Version: {expert.version}")

---

## 11. Streaming Responses

Waiting for a full response can feel slow, especially when the agent is searching docs, browsing the web, and running code. Streaming sends tokens as they're generated — the same experience you get in ChatGPT.

Let's stream a response from our multi-tool expert:

In [ ]:
# Stream a response from the multi-tool expert
stream = openai_client.responses.create(
    extra_body={"agent": {"name": "foundry-expert-full", "type": "agent_reference"}},
    input="Give me a quick overview of what tools are available for Foundry agents and when to use each one.",
    stream=True,
)

print("Agent (streaming): ", end="")
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
print()  # newline at the end

---

## 12. Clean Up Resources

When you're done experimenting, clean up the agents you created. This keeps your project tidy and avoids confusion.

In [ ]:
# Clean up all agents created in this tutorial
agents_to_clean = [
    ("foundry-expert", agent.version),
    ("foundry-expert-docs", docs_agent.version),
    ("foundry-expert-web", web_agent.version),
    ("foundry-expert-code", code_agent.version),
    ("foundry-expert-github", github_agent.version),
    ("foundry-expert-project", foundry_agent.version),
    ("foundry-expert-full", expert.version),
]

for name, version in agents_to_clean:
    try:
        project_client.agents.delete_version(agent_name=name, version=version)
        print(f"🗑️  Deleted: {name} v{version}")
    except Exception as e:
        print(f"⚠️  Could not delete {name} v{version}: {e}")

print("\n✅ Cleanup complete")

**Expected output:**
```
🗑️  Deleted: foundry-expert v1
🗑️  Deleted: foundry-expert-docs v1
🗑️  Deleted: foundry-expert-web v1
🗑️  Deleted: foundry-expert-code v1
🗑️  Deleted: foundry-expert-github v1
🗑️  Deleted: foundry-expert-project v1
🗑️  Deleted: foundry-expert-full v1

✅ Cleanup complete
```

---

## 13. Publish to Microsoft 365 Copilot & Teams

Your Foundry Expert works. Now let's put it where your team actually works — Microsoft 365 Copilot and Microsoft Teams.

### Publishing workflow

```
┌──────────────┐    ┌──────────────────┐    ┌─────────────────┐
│ Build & Test  │───▶│ Publish as Agent  │───▶│ Available in    │
│ in Foundry    │    │ Application       │    │ M365 Copilot    │
│               │    │                   │    │ & Teams         │
└──────────────┘    └──────────────────┘    └─────────────────┘
```

### Steps to publish

1. **Test thoroughly** in the Foundry portal or playground
2. **Publish** your agent as an Agent Application — in the Foundry portal, select your agent version, then click **Publish** → **Publish to Teams and Microsoft 365 Copilot**. An Azure Bot Service resource is created automatically.
3. **Configure metadata** — name, description, icons, and privacy policy
4. **Choose your distribution scope:**

| Scope | Visibility | Admin Approval | Best for |
|---|---|---|---|
| **Shared** | Your agents list | Not required | Personal testing, small teams |
| **Organization** | Built by your org | Required | Organization-wide deployment |

5. **Download the package** and upload to Teams for testing
6. For organization scope, **request admin approval** in the M365 admin center

### Required roles for publishing

| Action | Required Role |
|---|---|
| Publish agents | **Azure AI Project Manager** on Foundry project |
| Invoke/chat with published agents | **Azure AI User** on Agent Application |
| Approve org-wide distribution | **Microsoft 365 Admin** |

### Programmatic publishing (REST API)

If you prefer to automate publishing, you can use the REST API:

```bash
# 1. Create an Agent Application
PUT https://management.azure.com/subscriptions/{sub}/resourceGroups/{rg}/\
    providers/Microsoft.CognitiveServices/accounts/{account}/\
    projects/{project}/applications/{app_name}?api-version={version}

# 2. Create a Deployment
PUT https://management.azure.com/subscriptions/{sub}/resourceGroups/{rg}/\
    providers/Microsoft.CognitiveServices/accounts/{account}/\
    projects/{project}/applications/{app_name}/\
    agentdeployments/{deployment_name}?api-version={version}
```

> **Important:** A published agent gets its own **Agent Identity** in Microsoft Entra ID, separate from your project identity. Make sure to reassign RBAC permissions to the new identity for any Azure resources the agent accesses.

📚 Full guide: [Publish agents to Microsoft 365 Copilot and Teams](https://learn.microsoft.com/azure/ai-foundry/agents/how-to/publish-copilot)

---

## Next Steps

You've gone from zero to a multi-tool Foundry Expert with documentation search, web search, code execution, SDK review, and project-level intelligence — then streamed it and learned how to publish it to Teams.

Here's where to go next:

| Topic | What you'll learn |
|---|---|
| **[Function Calling](https://learn.microsoft.com/azure/ai-foundry/agents/how-to/tools/function-calling)** | Define custom functions your agent can invoke |
| **[Multi-Agent Workflows](https://learn.microsoft.com/azure/ai-foundry/agents/how-to/multi-agent-workflow)** | Orchestrate multiple agents with `WorkflowAgentDefinition` |
| **[Custom Code Interpreter](https://learn.microsoft.com/azure/ai-foundry/agents/how-to/tools/custom-code-interpreter)** | Deploy your own Python runtime with custom packages |
| **[MCP Authentication](https://learn.microsoft.com/azure/ai-foundry/agents/how-to/mcp-authentication)** | Secure MCP server connections |
| **[Tracing & Monitoring](https://learn.microsoft.com/azure/ai-foundry/agents/concepts/tracing)** | Set up Application Insights for observability |
| **[Evaluation](https://learn.microsoft.com/azure/ai-foundry/evaluation/evaluate-sdk)** | Measure agent quality with built-in evaluators |

### Useful links

- [Foundry Agent Service overview](https://learn.microsoft.com/azure/ai-foundry/agents/overview)
- [Environment setup guide](https://learn.microsoft.com/azure/ai-foundry/agents/environment-setup)
- [RBAC for Microsoft Foundry](https://learn.microsoft.com/azure/ai-foundry/concepts/rbac-foundry)
- [Azure AI Projects SDK reference](https://learn.microsoft.com/python/api/azure-ai-projects)
- [Foundry samples on GitHub](https://github.com/azure-ai-foundry/foundry-samples)

You built something real. Now go make it yours. 🔥